# Chapter 5 · Building an MCP server

The MCP server of Northside Veterinary Clinic, piece by piece: locating pets with errors that are flagged as errors, the lookup tools, the tools that act, and a resource. Next to it, a test suite that doesn't need the model. To measure what each one finds, ten defects are introduced into the server, one at a time, and the test suite is compared with the agent's suite of tasks.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the clinic

The test suite doesn't need the model; it is loaded further down, for the agent.

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # the versions the book was checked with

In [ ]:
"""Chapter 5: building an MCP server. The clinic server, piece by piece, with a test
suite that doesn't need the model."""
import asyncio
import copy
import json
import re
import time
import unicodedata
import urllib.request
from contextlib import AsyncExitStack
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/en/"
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; if not, on the CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def download(name):
    """A JSON file from the data folder; in Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
def reset():
    """Puts the clinic back as it was: the original schedule and no notices."""
    global APPOINTMENTS, NOTICES
    APPOINTMENTS, NOTICES = copy.deepcopy(APPOINTMENTS0), []


def strip_accents(text):
    """Lowercase and without accents, to compare names however they are written."""
    return "".join(c for c in unicodedata.normalize("NFD", str(text).lower())
                   if unicodedata.category(c) != "Mn").strip()


def notify(owner, text):
    """Sends a notice to the owner (here it is stored in a list)."""
    if owner not in OWNERS:
        return {"error": f"There is no client {owner}."}
    NOTICES.append({"owner": owner, "text": text})
    return {"sent": True, "to": OWNERS[owner]["name"],
            "via": OWNERS[owner]["notify_by"]}


def age(birth_date):
    """Full years on the date of the data."""
    b = date.fromisoformat(birth_date)
    return TODAY.year - b.year - ((TODAY.month, TODAY.day) < (b.month, b.day))


def pet_summary(pid):
    """The essentials of a pet, without the full history."""
    p = PETS[pid]
    return {"id": pid, "name": p["name"], "species": p["species"], "breed": p["breed"],
            "age_years": age(p["birth_date"]), "current_weight_kg": p["weight_kg"],
            "allergies": p["allergies"] or "none known",
            "owner": {"id": p["owner"], "name": OWNERS[p["owner"]]["name"]}}


def vaccines_with_status(pid):
    """Each vaccine with its next date and whether it is already overdue."""
    return [{"vaccine": v["vaccine"], "next": v["next"],
             "status": "overdue" if v["next"] < TODAY.isoformat() else "up to date"}
            for v in VACCINES[pid]]


def client_record_3(client_id):
    """A client's record with their pets and unpaid invoices (chapter 1, toolset 3)."""
    if client_id not in OWNERS:
        return {"error": f"There is no client {client_id}. Look them up by name with "
                         "search(table='owners', field='name', value=...)."}
    unpaid = [i for i in INVOICES if i["owner"] == client_id and not i["paid"]]
    return {"id": client_id, "name": OWNERS[client_id]["name"],
            "notify_by": OWNERS[client_id]["notify_by"],
            "pets": [{"id": p, "name": d["name"], "species": d["species"]}
                     for p, d in PETS.items() if d["owner"] == client_id],
            "unpaid_invoices": [{"id": i["id"], "date": i["date"], "total": i["total"]}
                                for i in unpaid],
            "total_unpaid": round(sum(i["total"] for i in unpaid), 2)}


def vet_of(name):
    """The vet's id from their name or their id."""
    for v, n in VETS.items():
        if strip_accents(name) in (strip_accents(v), strip_accents(n)) or \
                strip_accents(name) in strip_accents(n):
            return v
    return None


def new_appointments():
    """The appointments created during the task."""
    return [a for a in APPOINTMENTS if a["id"] not in {x["id"] for x in APPOINTMENTS0}]


def load_data():
    """Loads the clinic's data and leaves the clinic as it was at the start."""
    global PETS, OWNERS, HISTORY, VACCINES, APPOINTMENTS0, PHARMACY, INVOICES
    global VETS, TIMES
    PETS, OWNERS = download("pets.json"), download("owners.json")
    HISTORY, VACCINES = download("history.json"), download("vaccines.json")
    APPOINTMENTS0, PHARMACY = download("appointments.json"), download("pharmacy.json")
    INVOICES, VETS = download("invoices.json"), download("vets.json")
    TIMES = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reset()

In [ ]:
load_data()

## 2. The server, piece by piece

`create_server` builds the good server; each name in `DEFECTS` turns on one of the bugs that slip in when programming one.

In [ ]:
DEFECTS = {
    "accents": "compares the names with accents and capital letters",
    "silent_error": "a missing pet returns an empty result, without an error",
    "oldest_first": "the history starts with the oldest visit",
    "dose_without_allergies": "the dose does not warn about allergies",
    "vaccines_without_status": "the vaccines do not say whether they are overdue",
    "double_booking": "booking does not check whether the time is taken",
    "cancel_completed": "completed appointments can be canceled",
    "no_descriptions": "the parameters of free_slots have no description",
    "exception": "accounts fails with an exception if the client does not exist",
    "weekends": "free_slots offers times on Saturdays and Sundays",
}


Name = Annotated[str, Field(description="Name of the pet.")]


Owner = Annotated[str, Field(description="Owner's name; only if several pets have "
                                         "that name.")]


Date = Annotated[str, Field(description="YYYY-MM-DD.")]


Vet = Annotated[str, Field(description="Name of the vet.")]


def create_server(defects=frozenset()):
    """The clinic's MCP server: eleven tools and a resource."""
    server = MCPServer("clinic", instructions="Pets, vaccines, medical history, "
                       "schedule, notices and invoices of Northside Veterinary Clinic.")

    # Step 1: locate a pet, with errors that are flagged as errors
    def compare(text):
        """The text as it is compared: without accents, unless that defect is on."""
        return str(text) if "accents" in defects else strip_accents(text)

    def pet_of(name, owner=""):
        """The pet's id; a ToolError if there is none or there are several."""
        ids = [p for p, d in PETS.items() if compare(name) == compare(d["name"])
               and compare(owner) in compare(OWNERS[d["owner"]]["name"])]
        if not ids and "silent_error" in defects:
            return None
        if not ids:
            raise ToolError(f"There is no pet called \"{name}\""
                            + (f" belonging to {owner}" if owner else "")
                            + " at the clinic.")
        if len(ids) > 1:
            raise ToolError(f"There are {len(ids)} pets called \"{name}\"; give the "
                            "owner's name.")
        return ids[0]

    # Step 2: the lookup tools
    @server.tool()
    def find_pet(name: Name, owner: Owner = "") -> dict:
        """A pet's data, by name: species, age, weight, allergies and owner."""
        pid = pet_of(name, owner)
        return pet_summary(pid) if pid else {}

    @server.tool()
    def vaccines(pet: Name, owner: Owner = "") -> dict:
        """A pet's vaccines: the next date of each one and whether it is overdue."""
        pid = pet_of(pet, owner)
        if pid is None:
            return {"vaccines": []}
        listing = vaccines_with_status(pid)
        if "vaccines_without_status" in defects:
            listing = [{k: v for k, v in x.items() if k != "status"} for x in listing]
        return {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                "vaccines": listing}

    @server.tool()
    def history(pet: Name,
                last: Annotated[int, Field(description="How many visits, from the most "
                                                       "recent to the oldest.")] = 3,
                owner: Owner = "") -> dict:
        """A pet's last visits, starting with the most recent."""
        pid = pet_of(pet, owner)
        if pid is None:
            return {"visits": []}
        visits = [{k: v[k] for k in ("date", "reason", "treatment", "weight_kg")}
                  for v in HISTORY[pid][-last:]]
        if "oldest_first" not in defects:
            visits.reverse()
        return {"pet": PETS[pid]["name"], "visits": visits}

    @server.tool()
    def dose(medicine: Annotated[str, Field(description="Name of the medicine.")],
             pet: Name, owner: Owner = "") -> dict:
        """Dose of a medicine for the current weight, warning of allergy or species."""
        pid = pet_of(pet, owner)
        med = PHARMACY.get(strip_accents(medicine))
        if med is None:
            raise ToolError(f"We don't have \"{medicine}\". Medicines: "
                            + ", ".join(PHARMACY) + ".")
        if pid is None:
            return {}
        p, warnings = PETS[pid], []
        if strip_accents(medicine) in [strip_accents(a) for a in p["allergies"]] \
                and "dose_without_allergies" not in defects:
            warnings.append(f"{p['name']} is allergic to {medicine}: do not prescribe.")
        if p["species"] not in med["species"]:
            warnings.append(f"{medicine} is not indicated for {p['species']}s.")
        return {"medicine": strip_accents(medicine), "pet": p["name"],
                "weight_kg": p["weight_kg"],
                "dose_mg_per_dose": round(med["dose_mg_kg"] * p["weight_kg"], 2),
                "stock": med["stock"], "warnings": warnings or "none"}

    @server.tool()
    def appointments_of(pet: Name, owner: Owner = "") -> dict:
        """A pet's appointments (past and scheduled), with their id."""
        pid = pet_of(pet, owner)
        if pid is None:
            return {"appointments": []}
        return {"pet": PETS[pid]["name"], "today": TODAY.isoformat(), "appointments": [
            {"id": a["id"], "date": a["date"], "time": a["time"], "reason": a["reason"],
             "vet": VETS[a["vet"]], "status": a["status"]}
            for a in APPOINTMENTS if a["pet"] == pid]}

    if "no_descriptions" in defects:
        @server.tool()
        def free_slots(date: str, vet: str = "") -> dict:
            return free(date, vet)
    else:
        @server.tool()
        def free_slots(date: Date, vet: Annotated[str, Field(
                description="Name of the vet; empty for all.")] = "") -> dict:
            """Free times in the schedule on a weekday, for one vet or for all."""
            return free(date, vet)

    def free(day, vet):
        """The free times of a day; a ToolError for a bad date, a weekend or a vet."""
        try:
            when = date.fromisoformat(day)
        except ValueError:
            raise ToolError(f"\"{day}\" is not a date; use YYYY-MM-DD.")
        if when.weekday() >= 5 and "weekends" not in defects:
            raise ToolError(f"{day} is on a weekend and the clinic is closed.")
        v = vet_of(vet) if vet else None
        if vet and v is None:
            raise ToolError(f"There is no vet \"{vet}\". They are: "
                            + ", ".join(VETS.values()) + ".")
        taken = {(a["vet"], a["time"]) for a in APPOINTMENTS
                 if a["date"] == day and a["status"] != "canceled"}
        return {"date": day, "free_times": {
            VETS[x]: [t for t in TIMES if (x, t) not in taken]
            for x in ([v] if v else VETS)}}

    @server.tool()
    def accounts(owner: Annotated[str, Field(description="Owner's name.")]) -> dict:
        """Unpaid invoices of a client, looked up by name, and the total owed."""
        ids = [c for c, x in OWNERS.items()
               if strip_accents(owner) in strip_accents(x["name"])]
        if not ids and "exception" in defects:
            return client_record_3(next(iter(ids)))
        if not ids:
            raise ToolError(f"There is no client called \"{owner}\".")
        return {"clients": [{k: v for k, v in client_record_3(c).items() if k != "id"}
                            for c in ids]}

    # Step 3: the tools that act
    @server.tool()
    def book_appointment(pet: Name, vet: Vet, date: Date,
                         time: Annotated[str, Field(
                             description="HH:MM, on the hour or half past.")],
                         reason: Annotated[str, Field(
                             description="Reason for the appointment.")],
                         owner: Owner = "") -> dict:
        """Books an appointment at a free time."""
        pid, v = pet_of(pet, owner), vet_of(vet)
        if pid is None:
            return {}
        if v is None:
            raise ToolError(f"There is no vet \"{vet}\".")
        free(date, vet)                                 # checks the date and the day
        if time not in TIMES:
            raise ToolError(f"\"{time}\" is not an appointment time.")
        appointment = {"id": f"AP-{len(APPOINTMENTS0) + len(APPOINTMENTS) + 1:04d}",
                       "date": date, "time": time, "vet": v, "pet": pid,
                       "reason": reason, "status": "scheduled"}
        if "double_booking" not in defects and any(
                a["date"] == date and a["time"] == time and a["vet"] == v
                and a["status"] != "canceled" for a in APPOINTMENTS):
            raise ToolError(f"{VETS[v]} already has an appointment on {date} "
                            f"at {time}.")
        APPOINTMENTS.append(appointment)
        return {"booked": appointment["id"], "pet": PETS[pid]["name"], "vet": VETS[v],
                "date": date, "time": time}

    @server.tool()
    def cancel_appointment(appointment: Annotated[str, Field(
            description="Id of the appointment, like AP-0026 (given by "
                        "appointments_of).")]) -> dict:
        """Cancels a scheduled appointment."""
        for a in APPOINTMENTS:
            if a["id"] == appointment:
                if a["status"] != "scheduled" and "cancel_completed" not in defects:
                    raise ToolError(f"Appointment {appointment} is \"{a['status']}\".")
                a["status"] = "canceled"
                return {"canceled": appointment, "date": a["date"], "time": a["time"]}
        raise ToolError(f"There is no appointment {appointment}.")

    @server.tool()
    def notify_owner(pet: Name,
                     text: Annotated[str, Field(
                         description="Text of the notice, in English.")],
                     owner: Owner = "") -> dict:
        """Sends a notice to a pet's owner, by email or SMS as they prefer."""
        pid = pet_of(pet, owner)
        return notify(PETS[pid]["owner"], text) if pid else {}

    @server.tool()
    def pets_of(owner: Annotated[str, Field(description="Owner's name.")]) -> dict:
        """A client's pets, the client looked up by name."""
        ids = [c for c, x in OWNERS.items() if compare(owner) in compare(x["name"])]
        if not ids and "silent_error" not in defects:
            raise ToolError(f"There is no client called \"{owner}\".")
        return {"clients": [{"name": OWNERS[c]["name"], "pets": [
            {"name": x["name"], "species": x["species"]}
            for x in PETS.values() if x["owner"] == c]} for c in ids]}

    # Step 4: a resource, a document the application can read
    @server.resource("clinic://hours", description="Days and hours of the clinic's "
                     "appointments.")
    def hours() -> str:
        """The clinic's appointment hours and its vets."""
        return ("Appointments Monday to Friday, from 9 a.m. to 1:30 p.m. and from "
                "4 p.m. to 6:30 p.m., every half hour. Vets: "
                + ", ".join(VETS.values()) + ".")

    return server

In [ ]:
async def try_by_hand():
    async with Client(create_server()) as c:
        print([t.name for t in (await c.list_tools()).tools])
        print((await c.read_resource("clinic://hours")).contents[0].text)
        r = await c.call_tool("history", {"pet": "Zeus", "last": 2})
        print(r.is_error, r.content[0].text)
        r = await c.call_tool("vaccines", {"pet": "Buster"})
        print(r.is_error, r.content[0].text)

await try_by_hand()

## 3. The server's test suite

Twelve tests, one for each promise of the server. They take milliseconds.

In [ ]:
def content(result):
    """The data of a tool result: the JSON of its text block."""
    return json.loads(result.content[0].text)


async def test_described_schemas(c):
    """Every tool and every parameter has a description."""
    for t in (await c.list_tools()).tools:
        assert t.description, f"{t.name} has no description"
        for name, p in t.input_schema["properties"].items():
            assert p.get("description"), \
                f"the parameter {name} of {t.name} has no description"


async def test_one_line_descriptions(c):
    """No description has line breaks."""
    for t in (await c.list_tools()).tools:
        assert "\n" not in (t.description or ""), \
            f"the description of {t.name} has line breaks"


async def test_names_without_accents(c):
    """A pet is found however its name is written."""
    for name in ("Zoë", "zoe", "ZOE"):
        r = await c.call_tool("find_pet", {"name": name})
        assert not r.is_error and content(r).get("name") == "Zoë", \
            f"does not find \"{name}\""


async def test_pet_that_does_not_exist(c):
    """A pet that does not exist gives an error."""
    for tool in ("find_pet", "vaccines", "history", "appointments_of"):
        r = await c.call_tool(tool, {tool == "find_pet" and "name" or "pet": "Buster"})
        assert r.is_error, f"{tool} gives no error with a pet that does not exist"


async def test_history_most_recent_first(c):
    """The history starts with the most recent visit."""
    visits = content(await c.call_tool("history", {"pet": "Zeus", "last": 2}))["visits"]
    assert visits[0]["date"] > visits[1]["date"], \
        "the history does not start with the most recent visit"


async def test_dose_warns_of_allergies(c):
    """The dose warns about an allergy."""
    r = content(await c.call_tool("dose", {"medicine": "amoxicillin", "pet": "Simba"}))
    assert "allergic" in str(r["warnings"]), \
        "the dose of amoxicillin for Simba does not warn about her allergy"


async def test_vaccines_with_status(c):
    """The vaccines say which are overdue."""
    listing = content(await c.call_tool("vaccines", {"pet": "Toby"}))["vaccines"]
    assert [v.get("status") for v in listing].count("overdue") == 3, \
        "Toby's vaccines do not say that three are overdue"


async def test_does_not_book_a_taken_time(c):
    """Booking a taken time gives an error and leaves no appointment."""
    r = await c.call_tool("book_appointment", {"pet": "Toby", "vet": "Meyer",
                                               "date": "2026-10-22", "time": "09:00",
                                               "reason": "checkup"})
    assert r.is_error and not new_appointments(), "it books a time that is taken"


async def test_does_not_cancel_a_completed_appointment(c):
    """Canceling a completed appointment gives an error."""
    r = await c.call_tool("cancel_appointment", {"appointment": "AP-0008"})
    assert r.is_error, "it cancels an appointment that already took place"


async def test_no_slots_on_weekends(c):
    """There are no free times on a Saturday."""
    r = await c.call_tool("free_slots", {"date": "2026-10-24"})
    assert r.is_error, "it offers free times on a Saturday"


async def test_client_that_does_not_exist(c):
    """A client that does not exist gives an error that names them."""
    r = await c.call_tool("accounts", {"owner": "Nobody Nobody"})
    assert r.is_error and "Nobody" in r.content[0].text, \
        "the error for a client that does not exist does not explain it"


async def test_hours_resource(c):
    """The hours resource says the days of the appointments."""
    text = (await c.read_resource("clinic://hours")).contents[0].text
    assert "Monday to Friday" in text, "the hours resource does not say the days"


TESTS = [test_described_schemas, test_one_line_descriptions, test_names_without_accents,
         test_pet_that_does_not_exist, test_history_most_recent_first,
         test_dose_warns_of_allergies, test_vaccines_with_status,
         test_does_not_book_a_taken_time, test_does_not_cancel_a_completed_appointment,
         test_no_slots_on_weekends, test_client_that_does_not_exist,
         test_hours_resource]


async def test_server(server):
    """Runs all the tests, with the clinic reset before each one."""
    failures, start = {}, time.perf_counter()
    async with Client(server) as c:
        for test in TESTS:
            reset()
            try:
                await test(c)
            except AssertionError as e:
                failures[test.__name__] = str(e)
    return failures, round(time.perf_counter() - start, 2)

In [ ]:
for defect in [None] + list(DEFECTS):
    defects = frozenset([defect]) if defect else frozenset()
    failures, seconds = await test_server(create_server(defects))
    print(f"{defect or 'none':24s} {seconds} s", failures)

## 4. The agent's suite

Chapter 1's twelve tasks, with chapter 4's agent and compact host.

In [ ]:
tok, model = load(MODEL)

In [ ]:
SYSTEM = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in English, in one or two sentences.")


def generate(messages, schemas, max_tokens=400):
    """One answer from the model, and the input and output tokens."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    return (tok.decode(new, skip_special_tokens=True).strip(),
            inputs["input_ids"].shape[1], len(new))


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"], "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]


def date_in(answer, iso):
    """Does the date appear, as 2027-08-22, 8/22/2027, August 22, 2027 or 22 August?"""
    y, m, d = iso.split("-")
    month = MONTHS[int(m) - 1][:3]
    return says(answer, iso, rf"\b0?{int(m)}/0?{int(d)}/{y}",
                rf"\b{month}[a-z]*\.? 0?{int(d)}(st|nd|rd|th)?\b",
                rf"\b0?{int(d)}(st|nd|rd|th)? (of )?{month}")


def time_in(answer, hhmm):
    """Does the time appear, as 16:30, 4:30 p.m., 4:30pm or 4:30 (4 p.m. for 16:00)?"""
    h, m = (int(x) for x in hhmm.split(":"))
    h12, right, wrong = (h - 1) % 12 + 1, "pa"[h < 12], "ap"[h < 12]
    marker, not_wrong = rf"\s*{right}\.?\s?m\b", rf"(?!\s*{wrong}\.?\s?m\b)"
    forms = [rf"\b0?{h}:{m:02d}\b", rf"\b{h12}:{m:02d}\b{not_wrong}"]
    if m == 0:
        forms += [rf"\b{h12}{marker}",
                  rf"\b{h12}\b{not_wrong}(?=\s*(,|and\b|or\b|o'clock))"]
    if (h, m) == (12, 0):
        forms.append(r"\bnoon\b")
    return says(answer, *forms)


def first_free(date, vet):
    """The first free time of a vet on a day, in the original schedule."""
    taken = {a["time"] for a in APPOINTMENTS0 if a["date"] == date and a["vet"] == vet}
    return next(t for t in TIMES if t not in taken)


def owes(name):
    """What a client owes in unpaid invoices."""
    return round(sum(i["total"] for i in INVOICES if not i["paid"]
                     and OWNERS[i["owner"]]["name"] == name), 2)


def amount_in(answer, value):
    """Does the amount appear, as 136, 136.00, $136 or 136 dollars?"""
    whole = f"{value:.2f}".replace(".00", "").replace(".", r"\.")
    return says(answer, rf"\b{whole}(\.00?)?\b")


def notice_to(name, *forms):
    """Was a notice sent to that client, with all those forms in its text?"""
    return any(OWNERS[n["owner"]]["name"] == name
               and all(re.search(f, n["text"], re.I) for f in forms) for n in NOTICES)


TASKS = [
    ("rabies_vaccine", "When is Lola, Martha Fisher's dog, due for her next rabies "
     "vaccine?",
     lambda r: date_in(r, "2027-08-22")),
    ("dose", "How many milligrams of meloxicam should Max get per dose?",
     lambda r: says(r, r"2\.07", r"2\.1\b")),
    ("allergy", "They're going to prescribe amoxicillin for Simba. Is there any "
     "problem?",
     lambda r: says(r, r"allerg")),
    ("book", "Book a checkup for Toby with Dr. Paul Meyer on Thursday the 22nd, at the "
     "first free time.",
     lambda r: [(a["pet"], a["vet"], a["date"], a["time"]) for a in new_appointments()]
     == [("P-002", "V2", "2026-10-22", first_free("2026-10-22", "V2"))]),
    ("cancel", "Cancel Rocky's appointment tomorrow.",
     lambda r: [a["id"] for a in APPOINTMENTS if a["status"] == "canceled"]
     == ["AP-0060"]),
    ("overdue_notice", "Let Toby's owner know about his overdue vaccines.",
     lambda r: notice_to("Peter Adams", "rabies", "DHPP", "leptospirosis")),
    ("debt", "How much does Oscar Price owe in unpaid invoices?",
     lambda r: amount_in(r, owes("Oscar Price"))),
    ("marthas_cat", "Martha Fisher is calling about her cat. Is any of her vaccines "
     "overdue?",
     lambda r: says(r, "FVRCP")
     and says(r, "overdue|expired|past due|out of date|lapsed")),
    ("weight", "How much did Zeus weigh at his last visit, and how much has it changed "
     "since the one before?",
     lambda r: says(r, r"32\.2") and says(r, r"0\.6")
     and not says(r, r"(last|latest|most recent) visit[^.,]{0,30}(2025|31\.6)",
                  r"31\.6 kg (at|on|in|during) (his|the) (last|latest|most recent)")),
    ("last_visit", "Why did Willow come in last time, and what treatment did she get?",
     lambda r: says(r, "wound") and says(r, "amoxicillin")),
    ("free_slots", "What times is Dr. Anna Sutton free on Wednesday the 21st in the "
     "afternoon?",
     lambda r: all(time_in(r, t) for t in ("16:00", "16:30", "17:30", "18:00", "18:30"))
     and not time_in(r, "17:00")),
    ("does_not_exist", "When is Buster due for his rabies vaccine?",
     lambda r: says(r, r"\bno (pet|patient|record|animal|such|one|information)\b",
                    r"(n't|not|unable to) (exist|find|locate|appear|registered)",
                    r"\bnot (found|a patient|registered|on file|in our)",
                    r"\bthere (is|are) no\b", r"\bnone\b")
     and not says(r, r"\b20\d\d-\d\d-\d\d\b", r"\b\d\d?/\d\d?/20\d\d\b")),
]

In [ ]:
def without_titles(schema):
    """The schema without the "title" keys that Pydantic adds (the model needs none)."""
    if isinstance(schema, dict):
        return {k: without_titles(v) for k, v in schema.items() if k != "title"}
    return schema


def compact(text):
    """A result's JSON on a single line, without the indentation the server sends."""
    try:
        return json.dumps(json.loads(text), ensure_ascii=False)
    except ValueError:
        return text


class Host:
    """Connects to several MCP servers, gathers their tools and sends each call to the
    server that offers it. It knows nothing about the clinic. With compact, it removes
    the titles from the schemas and writes the results on a single line."""

    def __init__(self, servers, compact=False):
        self.servers, self.compact = servers, compact

    async def __aenter__(self):
        self.stack = AsyncExitStack()
        self.route, self.schemas = {}, []
        for server in self.servers:
            client = await self.stack.enter_async_context(Client(server))
            for t in (await client.list_tools()).tools:
                self.route[t.name] = client
                parameters = without_titles(t.input_schema) if self.compact \
                    else t.input_schema
                self.schemas.append({"type": "function", "function": {
                    "name": t.name, "description": t.description,
                    "parameters": parameters}})
        return self

    async def __aexit__(self, *error):
        await self.stack.aclose()

    async def call(self, name, arguments):
        """Calls the tool on its server and returns the text of the result."""
        client = self.route.get(name)
        if client is None:
            return json.dumps({"error": f"There is no tool {name}."},
                              ensure_ascii=False)
        r = await client.call_tool(name, arguments)
        text = "\n".join(b.text for b in r.content if b.type == "text")
        return compact(text) if self.compact else text


async def mcp_agent(task, host, max_steps=8, show=False):
    """Chapter 1's loop, with the tools of the MCP servers."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_in, n_out = generate(messages, host.schemas)
        calls = read_calls(answer)[:1]
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "thought": thought,
                      "calls": []})
        if not calls:
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            text = await host.call(call["name"], call["arguments"])
            trace[-1]["calls"].append({**call, "result": text,
                                       "result_tokens": len(tok(text)["input_ids"])})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                args = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({args})")
                print(f"     → {text[:150]}")
    return None, trace


async def test_agent(server, tasks=TASKS):
    """The tasks with chapter 4's agent and a compact host connected to the server."""
    rows, start = [], time.perf_counter()
    async with Host([server], compact=True) as host:
        for key, task, check in tasks:
            reset()
            answer, trace = await mcp_agent(task, host)
            rows.append({"task": key, "ok": bool(check(answer)), "answer": answer,
                         "calls": [(c["name"], c["arguments"]) for s in trace
                                   for c in s["calls"]]})
    return rows, round(time.perf_counter() - start, 1)

In [ ]:
rows, seconds = await test_agent(create_server())
print(sum(r["ok"] for r in rows), "of 12 in", seconds, "s")
rows, seconds = await test_agent(create_server(frozenset(["oldest_first"])))
print(sum(r["ok"] for r in rows), "of 12 in", seconds, "s with the history reversed")

## Exercises

1. Explain in your own words the difference between returning `{"error": …}` and raising `ToolError`. What does the client see in each case?

2. Write one more test for each tool that acts: that a correct booking appears in the schedule and that a correct cancellation changes the appointment's status.

3. Turn on two defects at once, `accents` and `oldest_first`. Do the two expected tests fail? Does any other?

4. Make up an eleventh defect that the test suite doesn't find. Which test would have to be added? Does the agent find it?

5. Measure how many tokens each result of the chapter's server takes, and write a test that fails if any goes over 300.

6. Add a second resource, `clinic://prices`, with the clinic's prices, and its test. How does the client read it?

7. Change `history` so that it returns a Pydantic model instead of a dictionary. What changes in `tools/list`? And in the result of `call_tool`?

8. With the `double_booking` defect, the agent doesn't lose any task. Write an agent task that does uncover it. How long does it take to uncover it, compared with the test?

9. Run the test suite on chapter 4's pharmacy server, through stdio. Which tests make sense for it? Write three.

10. **Project.** Write a test suite for the server of your chapter 4 project: at least one schema test, one with known results, one for errors and one for state. Introduce three defects and check that the tests find them.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [MCP specification](https://modelcontextprotocol.io/specification) and [Python SDK](https://github.com/modelcontextprotocol/python-sdk). Northside Veterinary Clinic and its data are fictional. Original code and experiments.